In [1]:
import os
import numpy as np
import pickle
import torch
from sklearn.decomposition import IncrementalPCA, PCA
import transformers
from transformers import AutoTokenizer, EsmForProteinFolding

In [2]:
fpath = r'C:\Users\valle\PycharmProjects\Landscape\Pipeline\STR_cluster\test_structures_'

In [3]:
all_data = {}
for file in os.listdir(fpath):
    with open(os.path.join(fpath, file), "rb") as f:
        structure = pickle.load(f)
        variant = list(structure.keys())[0]
        all_data.update({variant: {"Structure": structure[variant]["Structure"], "Fitness": structure[variant]["Fitness"]}})

In [4]:
all_data.keys()
list_keys_all_data = list(all_data.keys())
list_keys_all_data

['VDGA', 'VDGC', 'VDGD', 'VDGT', 'VDGY']

In [5]:
all_data[list_keys_all_data[0]]["Structure"].keys()

odict_keys(['frames', 'sidechain_frames', 'unnormalized_angles', 'angles', 'positions', 'states', 's_s', 's_z', 'distogram_logits', 'lm_logits', 'aatype', 'atom14_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'atom37_atom_exists', 'residue_index', 'lddt_head', 'plddt', 'ptm_logits', 'ptm', 'aligned_confidence_probs', 'predicted_aligned_error', 'max_predicted_aligned_error'])

In [6]:
all_data[list_keys_all_data[0]]["Structure"]['positions'].shape

torch.Size([8, 1, 56, 14, 3])

In [27]:
from transformers.models.esm.openfold_utils.protein import to_pdb, Protein as OFProtein
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37

def convert_outputs_to_pdb(outputs):
    final_atom_positions = atom14_to_atom37(outputs["positions"][-1], outputs)
    outputs = {k: v.to("cpu").detach().numpy() for k, v in outputs.items()}
    final_atom_positions = final_atom_positions.cpu().detach().numpy()
    final_atom_mask = outputs["atom37_atom_exists"]
    pdbs = []
    for i in range(outputs["aatype"].shape[0]):
        aa = outputs["aatype"][i]
        pred_pos = final_atom_positions[i]
        mask = final_atom_mask[i]
        resid = outputs["residue_index"][i] + 1
        pred = OFProtein(
            aatype=aa,
            atom_positions=pred_pos,
            atom_mask=mask,
            residue_index=resid,
            b_factors=outputs["plddt"][i],
            chain_index=outputs["chain_index"][i] if "chain_index" in outputs else None,
        )
        pdbs.append(to_pdb(pred))
    return pdbs

In [43]:
pdb = convert_outputs_to_pdb(all_data[list_keys_all_data[0]]["Structure"])

In [47]:
pdb[0]

'PARENT N/A\nATOM      1  N   MET A   1       1.493  -1.674 -12.812  1.00  0.90           N  \nATOM      2  CA  MET A   1       0.135  -1.634 -12.279  1.00  0.91           C  \nATOM      3  C   MET A   1       0.048  -0.695 -11.080  1.00  0.92           C  \nATOM      4  CB  MET A   1      -0.328  -3.036 -11.879  1.00  0.88           C  \nATOM      5  O   MET A   1       1.058  -0.415 -10.431  1.00  0.91           O  \nATOM      6  CG  MET A   1      -0.522  -3.977 -13.058  1.00  0.76           C  \nATOM      7  SD  MET A   1      -0.862  -5.702 -12.532  1.00  0.71           S  \nATOM      8  CE  MET A   1       0.833  -6.322 -12.353  1.00  0.67           C  \nATOM      9  N   THR A   2      -1.036  -0.101 -10.874  1.00  0.94           N  \nATOM     10  CA  THR A   2      -1.261   0.773  -9.728  1.00  0.95           C  \nATOM     11  C   THR A   2      -1.761  -0.027  -8.528  1.00  0.94           C  \nATOM     12  CB  THR A   2      -2.272   1.884 -10.064  1.00  0.93           C  \nATO

In [52]:
import py3Dmol
    
view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=800, height=400)
view.addModel("".join(pdb), 'pdb')
view.setStyle({'model': -1}, {"stick": {'color': 'spectrum'}})

You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol

In [35]:
from Bio.PDB import PDBParser, PDBIO

In [ ]:
pdb

In [33]:
with open(r"C:\Users\valle\PycharmProjects\Landscape\Pipeline\STR_cluster\saved_pdb\test.pdb", "w") as f:
    f.write(pdb)

TypeError: write() argument must be str, not list

In [ ]:
import py3Dmol

view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=800, height=400)
view.addModel("".join(pdb), 'pdb')
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})